# Unsloth GRPO Training for Hierarchical Reasoning

This notebook uses Unsloth's optimized kernels with TRL's GRPOTrainer for stable training.

**Key features:**
- ~50% less VRAM usage than standard transformers
- vLLM fast inference for generation
- HICRA-inspired reward functions for reasoning

In [1]:
# Cell 1: Environment Setup
import os
os.environ["UNSLOTH_VLLM_STANDBY"] = "1"  # Extra 30% context lengths
os.environ["fix_mistral_regex"] = "True"
os.environ["OMP_NUM_THREADS"] = "1"

# Install dependencies (run this if not already installed)
# !pip install unsloth vllm
# !pip install transformers==4.56.2
# !pip install --no-deps trl==0.22.2

In [3]:
# In the Space terminal
# conda install python=3.10 -y
% pip install --upgrade unsloth

UsageError: Line magic function `%` not found.


In [4]:

# OR Install required packages (needed after Space restarts)
%pip install -q huggingface_hub unsloth bitsandbytes datasets dotenv
%pip install httpx==0.27.2

/home/user/miniconda/bin/python: No module named pip
Note: you may need to restart the kernel to use updated packages.
/home/user/miniconda/bin/python: No module named pip
Note: you may need to restart the kernel to use updated packages.


In [5]:
# Set remote HF_TOKEN from local .env
import os
from dotenv import load_dotenv

load_dotenv()
hf_token = os.getenv('HF_TOKEN')

# ssh -i ~/.ssh/id_ed25519 dataimaginations-heirarchical-reasoning@ssh.hf.space "echo 'export HF_TOKEN={hf_token}' >> ~/.bashrc"
print("✅ Token set! Restart remote shell to activate.")

✅ Token set! Restart remote shell to activate.


In [7]:
import os
from huggingface_hub import login

# Login using your HF token
hf_token = os.getenv('HF_TOKEN')  # Try environment variable first

if hf_token:
    # login(token=hf_token)
    login()
    print("✅ Logged in with HF_TOKEN environment variable")
else:
    # If no env var, prompt for token (you'll need to paste it)
    login()

✅ Logged in with HF_TOKEN environment variable


In [9]:
from unsloth import FastLanguageModel
import torch
# Configuration
max_seq_length = 2048  # Can increase for longer reasoning traces
lora_rank = 32  # Larger rank = smarter, but slower
print("⏳ Loading model with Unsloth...")
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name="unsloth/Qwen3-VL-8B-Instruct-unsloth-bnb-4bit",
    max_seq_length=max_seq_length,
    load_in_4bit=True,
    fast_inference=False,  # Disabled - requires CUDA toolkit for vLLM
)
print("🔗 Attaching LoRA adapters...")
model = FastLanguageModel.get_peft_model(
    model,
    r=lora_rank,
    target_modules=[
        "q_proj", "k_proj", "v_proj", "o_proj",
        "gate_proj", "up_proj", "down_proj",
    ],
    lora_alpha=lora_rank,
    use_gradient_checkpointing="unsloth",  # Optimized gradient checkpointing
    random_state=3407,
)
print("✅ Model loaded successfully!")

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.


/tmp/ipykernel_681/3583319321.py:1: UserWarning: WARNING: Unsloth should be imported before [transformers] to ensure all optimizations are applied. Your code may run slower or encounter memory issues without these optimizations.

Please restructure your imports with 'import unsloth' at the top of your file.
  from unsloth import FastLanguageModel


SyntaxError: invalid syntax (gpt_oss.py, line 1303)

Tuning Tips:
- If you're getting OOM: Lower NEMOTRON_SAMPLE_SIZE to 1000-2000
- If generations are too long: Lower MAX_ANSWER_TOKENS to 400
- If you want more data: Increase NEMOTRON_SAMPLE_SIZE to 5000+
The filtering keeps ~60-70% of examples typically, so 3000 samples → ~2000 usable examples mixed with your 729 HICRA examples.



In [ ]:
# Cell 4: Load and Combine Datasets
from datasets import load_dataset, Dataset
import json

# === Configuration ===
MAX_PROMPT_TOKENS = 400    # Filter out prompts longer than this
MAX_ANSWER_TOKENS = 600    # Filter out answers longer than this  
NEMOTRON_SAMPLE_SIZE = 3000  # How many Nemotron examples to use

# System prompt for reasoning format
SYSTEM_PROMPT = """
You are a mathematical reasoning assistant. Think through problems step by step.
Respond in the following format:
<think>
...
</think>
<answer>
...
</answer>
"""

def format_prompt(example):
    """Format dataset for GRPO training with chat template."""
    return {
        'prompt': [
            {'role': 'system', 'content': SYSTEM_PROMPT.strip()},
            {'role': 'user', 'content': example['prompt']}
        ],
        'answer': str(example['answer'])
    }

def format_nemotron(example):
    """Convert Nemotron format to our format."""
    messages = example.get('messages', [])
    
    # Extract user prompt and assistant answer
    user_content = ""
    assistant_content = ""
    
    for msg in messages:
        if msg['role'] == 'user':
            user_content = msg['content']
        elif msg['role'] == 'assistant':
            assistant_content = msg['content']
    
    # Get expected answer (fallback to assistant content if not available)
    expected = example.get('expected_answer', '')
    if not expected:
        # Try to extract from assistant's <answer> tags if present
        if '<answer>' in assistant_content and '</answer>' in assistant_content:
            expected = assistant_content.split('<answer>')[-1].split('</answer>')[0].strip()
        else:
            expected = assistant_content[-200:] if len(assistant_content) > 200 else assistant_content
    
    return {
        'prompt': user_content,
        'answer': str(expected)
    }

def estimate_tokens(text):
    """Rough token estimate (1 token ≈ 4 chars for English)."""
    return len(str(text)) // 4

def filter_by_length(example):
    """Filter out examples that are too long."""
    prompt_tokens = estimate_tokens(example['prompt'])
    answer_tokens = estimate_tokens(example['answer'])
    return prompt_tokens <= MAX_PROMPT_TOKENS and answer_tokens <= MAX_ANSWER_TOKENS

# === 1. Load Your HICRA Synthetic Data ===
print("📂 Loading HICRA dataset...")
my_dataset = load_dataset(
    "json", 
    data_files="reasoning_dataset_v2_train.json", 
    split="train"
)
print(f"   ✅ Loaded {len(my_dataset)} HICRA examples")

# === 2. Load Nemotron Math Data (Streaming) ===
print(f"🌊 Streaming {NEMOTRON_SAMPLE_SIZE} Nemotron math examples...")
try:
    nemotron_stream = load_dataset(
        "nvidia/Nemotron-Post-Training-Dataset-v1", 
        split="math", 
        streaming=True
    )
    
    # Take a sample and convert to list
    nemotron_list = []
    for i, example in enumerate(nemotron_stream):
        if i >= NEMOTRON_SAMPLE_SIZE:
            break
        formatted = format_nemotron(example)
        # Only keep if it's not too long
        if filter_by_length(formatted):
            nemotron_list.append(formatted)
        
        if (i + 1) % 500 == 0:
            print(f"   Processed {i + 1} examples, kept {len(nemotron_list)}...")
    
    nemotron_dataset = Dataset.from_list(nemotron_list)
    print(f"   ✅ Loaded {len(nemotron_dataset)} Nemotron examples (after length filter)")
    
except Exception as e:
    print(f"   ⚠️ Could not load Nemotron: {e}")
    print("   Continuing with HICRA data only...")
    nemotron_dataset = None

# === 3. Combine Datasets ===
print("🔀 Combining datasets...")

# Filter HICRA by length too
my_dataset_filtered = my_dataset.filter(filter_by_length)
print(f"   HICRA after filter: {len(my_dataset_filtered)} examples")

if nemotron_dataset and len(nemotron_dataset) > 0:
    from datasets import concatenate_datasets
    
    # Make sure both have the same columns
    combined_dataset = concatenate_datasets([my_dataset_filtered, nemotron_dataset])
    print(f"   ✅ Combined dataset: {len(combined_dataset)} examples")
else:
    combined_dataset = my_dataset_filtered
    print(f"   ✅ Using HICRA only: {len(combined_dataset)} examples")

# === 4. Format for GRPO Training ===
print("📝 Formatting for GRPO...")
dataset_train = combined_dataset.map(format_prompt)

# Shuffle to mix the datasets
dataset_train = dataset_train.shuffle(seed=42)

# === 5. Load Test Set (HICRA only) ===
dataset_test = load_dataset(
    "json", 
    data_files="reasoning_dataset_v2_test.json", 
    split="train"
).map(format_prompt)

print(f"\n✅ Final Training Set: {len(dataset_train)} examples")
print(f"✅ Test Set: {len(dataset_test)} examples")
print(f"\nSample prompt format:")
print(dataset_train[0]['prompt'])

📂 Loading HICRA dataset...
   ✅ Loaded 729 HICRA examples
🌊 Streaming 3000 Nemotron math examples...
   Processed 500 examples, kept 498...
   Processed 1000 examples, kept 998...
   Processed 1500 examples, kept 1498...
   Processed 2000 examples, kept 1998...
   Processed 2500 examples, kept 2498...
   Processed 3000 examples, kept 2997...
   ✅ Loaded 2997 Nemotron examples (after length filter)
🔀 Combining datasets...


Filter: 100%|██████████| 729/729 [00:00<00:00, 48106.48 examples/s]


   HICRA after filter: 729 examples
   ✅ Combined dataset: 3726 examples
📝 Formatting for GRPO...


Map: 100%|██████████| 3726/3726 [00:00<00:00, 17063.55 examples/s]



✅ Final Training Set: 3726 examples
✅ Test Set: 36 examples

Sample prompt format:
[{'content': 'You are a mathematical reasoning assistant. Think through problems step by step.\nRespond in the following format:\n<reasoning>\n...\n</reasoning>\n<answer>\n...\n</answer>', 'role': 'system'}, {'content': 'Evaluate the integral \\(\\int_0^{2\\pi} \\sqrt{\\sin^2(t) \\cos^2(t)} \\, dt\\).', 'role': 'user'}]


In [5]:
# Cell 5: Reward Functions
import re

# Strategic reasoning phrases (from HICRA paper)
STRATEGIC_GRAMS = [
    "first i need to", "let's look at", "alternatively", "wait",
    "but i'm not sure", "let's see if", "notice that",
    "the final answer is", "let's assume", "we can conclude",
    "implies that", "to solve this", "break it down",
    "suppose that", "checking the", "recall that",
    "step 1", "step 2", "therefore", "thus"
]

def extract_xml_answer(text: str) -> str:
    """Extract answer from <answer> tags."""
    if "<answer>" not in text:
        return text.strip()
    answer = text.split("<answer>")[-1]
    answer = answer.split("</answer>")[0]
    return answer.strip()

def correctness_reward_func(prompts, completions, answer, **kwargs) -> list[float]:
    """
    Check if the model's answer matches the expected answer.
    Returns 2.0 for correct, 0.0 for incorrect.
    """
    responses = [completion[0]['content'] for completion in completions]
    extracted = [extract_xml_answer(r) for r in responses]
    
    # Debug output (first item only)
    q = prompts[0][-1]['content'][:100]  # First 100 chars of question
    print(f"---\nQ: {q}...\nExpected: {answer[0]}\nExtracted: {extracted[0][:50]}...")
    
    rewards = []
    for ext, ans in zip(extracted, answer):
        # Check if answer appears in extracted text
        if str(ans).strip() in ext:
            rewards.append(2.0)
        else:
            rewards.append(0.0)
    return rewards

def reasoning_reward_func(completions, **kwargs) -> list[float]:
    """
    HICRA-inspired reward for reasoning structure.
    Gives bonus for using strategic reasoning phrases.
    """
    responses = [completion[0]['content'] for completion in completions]
    rewards = []
    
    for response in responses:
        score = 0.0
        response_lower = response.lower()
        
        # Check for strategic grams
        for gram in STRATEGIC_GRAMS:
            if gram in response_lower:
                score += 0.05
        
        # Bonus for using reasoning tags
        if "<reasoning>" in response and "</reasoning>" in response:
            score += 0.2
        if "<answer>" in response and "</answer>" in response:
            score += 0.1
        
        # Cap the reward
        rewards.append(min(score, 0.5))
    
    return rewards

def format_reward_func(completions, **kwargs) -> list[float]:
    """Reward for correct XML format."""
    pattern = r"<reasoning>.*?</reasoning>\s*<answer>.*?</answer>"
    responses = [completion[0]['content'] for completion in completions]
    return [0.5 if re.search(pattern, r, re.DOTALL) else 0.0 for r in responses]

print("✅ Reward functions defined")

✅ Reward functions defined


### Chat Template (Save for Base models)

```
# Set Llama 3 chat template (required for GRPO with conversational data)
tokenizer.chat_template = """{% for message in messages %}{% if message['role'] == 'system' %}<|begin_of_text|><|start_header_id|>system<|end_header_id|>
{{ message['content'] }}<|eot_id|>{% elif message['role'] == 'user' %}<|start_header_id|>user<|end_header_id|}
{{ message['content'] }}<|eot_id|>{% elif message['role'] == 'assistant' %}<|start_header_id|>assistant<|end_header_id|>
{{ message['content'] }}<|eot_id|>{% endif %}{% endfor %}{% if add_generation_prompt %}<|start_header_id|>assistant<|end_header_id|>
{% endif %}"""
print("✅ Chat template set!")
```

**Optional: Use More GPU**
You could also try:

- `num_generations=6` (more diverse rollouts per step)
- Or increase `max_seq_length=1280` in cell 3 if Nemotron answers are very long

In [9]:
# Cell 7 (updated)
from trl import GRPOConfig, GRPOTrainer

# Adjusted: Give more tokens to completions
max_prompt_length = 384     # Up from 256 - plenty for most math questions
max_completion_length = 640 # 1024 - 384 = 640 tokens for reasoning

training_args = GRPOConfig(
    # Optimization
    learning_rate=5e-6,
    adam_beta1=0.9,
    adam_beta2=0.99,
    weight_decay=0.1,
    warmup_ratio=0.1,
    lr_scheduler_type="cosine",
    optim="paged_adamw_8bit",
    
    # Batch settings - you have headroom!
    per_device_train_batch_size=1,
    gradient_accumulation_steps=4,
    num_generations=4,  # Could try 6 if you want more diversity
    
    # Sequence lengths (adjusted)
    max_prompt_length=max_prompt_length,
    max_completion_length=max_completion_length,
    
    # Training duration
    max_steps=500,  # Bumped up for larger dataset
    
    # Stability
    max_grad_norm=0.1,
    
    # Logging & Saving
    logging_steps=1,
    save_steps=100,
    output_dir="llama-1b-reasoning-unsloth-v2",
    report_to="none",
)

print(f"✅ Training configuration set")
print(f"   Prompt: {max_prompt_length} tokens, Completion: {max_completion_length} tokens")

✅ Training configuration set
   Prompt: 384 tokens, Completion: 640 tokens


In [10]:
# Cell 7: Initialize Trainer
print("🚀 Initializing GRPO Trainer...")

trainer = GRPOTrainer(
    model=model,
    processing_class=tokenizer,
    reward_funcs=[
        correctness_reward_func,
        reasoning_reward_func,
        format_reward_func,
    ],
    args=training_args,
    train_dataset=dataset_train,
)

print("✅ Trainer initialized!")

🚀 Initializing GRPO Trainer...
✅ Trainer initialized!


Current settings `nvidia-smi` says: `6861MiB /  12282MiB`

In [ ]:
# Cell 8: Run Training!
print("🏋️ Starting training...")
print("Note: First ~100 steps may show 0 reward. Be patient!")
print("="*50)

trainer_stats = trainer.train()

print("="*50)
print("✅ Training complete!")

The model is already on multiple devices. Skipping the move to device specified in `args`.


🏋️ Starting training...
Note: First ~100 steps may show 0 reward. Be patient!


==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 3,726 | Num Epochs = 1 | Total steps = 500
O^O/ \_/ \    Batch size per device = 1 | Gradient accumulation steps = 4
\        /    Data Parallel GPUs = 1 | Total batch size (1 x 4 x 1) = 4
 "-____-"     Trainable parameters = 22,544,384 of 1,258,358,784 (1.79% trained)


Unsloth: Will smartly offload gradients to save VRAM!
---
Q: Solve the system of linear equations:
\[ 3x - 5y + 4z = 5 \]
\[ 7x + 2y - 3z = 2 \]
\[ 4x + 3y - 7z ...
Expected: 3) = 7 + 4 - 9 = 2\), which holds.
- For \(4x + 3y - 7z = -11\): \(4(1) + 3(2) - 7(3) = 4 + 6 - 21 = -11\), which holds.

The solution is presented as an ordered triple \((x, y, z)\).

\boxed{(1,2,3)}
Extracted: To solve this system of linear equations, we can u...


Step,Training Loss,reward,reward_std,completions / mean_length,completions / min_length,completions / max_length,completions / clipped_ratio,completions / mean_terminated_length,completions / min_terminated_length,completions / max_terminated_length,sampling / sampling_logp_difference / mean,sampling / sampling_logp_difference / max,sampling / importance_sampling_ratio / min,sampling / importance_sampling_ratio / mean,sampling / importance_sampling_ratio / max,kl,rewards / correctness_reward_func / mean,rewards / correctness_reward_func / std,rewards / reasoning_reward_func / mean,rewards / reasoning_reward_func / std,rewards / format_reward_func / mean,rewards / format_reward_func / std
1,0.000000,0.087500,0.085391,640.000000,640.000000,640.000000,1.000000,0.000000,0.000000,0.000000,0,0,0,0,0,0.000328,0.000000,0.000000,0.087500,0.085391,0.000000,0.000000
2,0.000000,0.037500,0.047871,484.000000,260.000000,640.000000,0.500000,328.000000,260.000000,396.000000,No Log,No Log,No Log,No Log,No Log,0.000508,0.000000,0.000000,0.037500,0.047871,0.000000,0.000000
3,0.000000,0.050000,0.040825,494.750000,292.000000,640.000000,0.500000,349.500000,292.000000,407.000000,No Log,No Log,No Log,No Log,No Log,0.000539,0.000000,0.000000,0.050000,0.040825,0.000000,0.000000
4,0.000000,0.037500,0.025000,585.000000,420.000000,640.000000,0.750000,420.000000,420.000000,420.000000,No Log,No Log,No Log,No Log,No Log,0.000440,0.000000,0.000000,0.037500,0.025000,0.000000,0.000000
5,0.000000,0.037500,0.047871,409.500000,132.000000,640.000000,0.500000,179.000000,132.000000,226.000000,No Log,No Log,No Log,No Log,No Log,0.000507,0.000000,0.000000,0.037500,0.047871,0.000000,0.000000
6,0.000000,0.012500,0.025000,361.250000,202.000000,640.000000,0.250000,268.333344,202.000000,368.000000,No Log,No Log,No Log,No Log,No Log,0.000323,0.000000,0.000000,0.012500,0.025000,0.000000,0.000000
7,0.000000,0.000000,0.000000,640.000000,640.000000,640.000000,1.000000,0.000000,0.000000,0.000000,No Log,No Log,No Log,No Log,No Log,0.000464,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
8,0.000000,0.025000,0.028868,370.750000,178.000000,640.000000,0.250000,281.000000,178.000000,439.000000,No Log,No Log,No Log,No Log,No Log,0.000774,0.000000,0.000000,0.025000,0.028868,0.000000,0.000000
9,0.000000,2.125000,0.086603,554.000000,296.000000,640.000000,0.750000,296.000000,296.000000,296.000000,No Log,No Log,No Log,No Log,No Log,0.000412,2.000000,0.000000,0.125000,0.086603,0.000000,0.000000
10,0.000000,0.150000,0.091287,544.000000,333.000000,640.000000,0.500000,448.000000,333.000000,563.000000,No Log,No Log,No Log,No Log,No Log,0.000461,0.000000,0.000000,0.150000,0.091287,0.000000,0.000000


---
Q: Solve the given equations: $\sin (a+x)+\sin x=\cos \frac{a}{2}$....
Expected: that case.

The solution for \(x\) in terms of \(a\) is:
\[
\boxed{x = -\dfrac{a}{2} + \dfrac{\pi}{6} + 2k\pi \quad \text{or} \quad x = -\dfrac{a}{2} + \dfrac{5\pi}{6} + 2k\pi, \quad k \in \mathbb{Z}}
Extracted: To solve this equation, we can start by applying t...
---
Q: In a cube with edge a through the midpoints of two parallel edges not lying in one face a straight l...
Expected: sidering the symmetry and using integration or geometric properties. The expression accounts for the specific rotation axis and the cube's dimensions.

\boxed{\dfrac{a^{3}}{3}\left(3\sqrt{2}-2\right)}
Extracted: To find the volume of the common portion, let's st...
---
Q: A quality control manager oversees three production lines that produce defective items at rates of 2...
Expected: 1182
Extracted: To solve this problem, we need to find a distribut...
---
Q: Triangle ABC has sides AC = 3, BC = 5, and AB = 7. A circle is d

In [9]:
# Cell 9: Save Model
import os

# Option 1: Save locally
output_path = "llama-1b-reasoning-unsloth-HICRA-v1"
model.save_pretrained(output_path)
tokenizer.save_pretrained(output_path)
print(f"✅ Model saved to {output_path}")

# Option 2: Push to HuggingFace Hub (uncomment to use)
# repo_name = "DataImaginations/Llama-1B-Reasoning-v1"
# hf_token = os.getenv('HF_TOKEN')
# 
# print(f"⏳ Pushing to {repo_name}...")
# model.push_to_hub_merged(
#     repo_name,
#     tokenizer,
#     save_method="merged_16bit",
#     token=hf_token
# )
# print("✅ Model pushed to Hub!")

✅ Model saved to llama-1b-reasoning-unsloth-HICRA-v1


In [1]:
# Cell: Merge LoRA adapters and save for evaluation
from unsloth import FastLanguageModel

# Load the adapter model
model, tokenizer = FastLanguageModel.from_pretrained(
    "llama-1b-reasoning-unsloth-HICRA-v1",
    max_seq_length=1024,
    load_in_4bit=True,
)

# Merge and save in 16-bit
print("⏳ Merging adapters...")
model.save_pretrained_merged(
    "llama-1b-reasoning-merged",  # New path for merged model
    tokenizer,
    save_method="merged_16bit",  # Full precision merged weights
)
print("✅ Merged model saved!")

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.


/home/david-barnes/Documents/Projects/heirarch/.venv/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


🦥 Unsloth Zoo will now patch everything to make training faster!
==((====))==  Unsloth 2025.12.8: Fast Llama patching. Transformers: 4.57.3. vLLM: 0.13.0.
   \\   /|    NVIDIA GeForce RTX 4070 SUPER. Num GPUs = 1. Max memory: 11.594 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.9.0+cu128. CUDA: 8.9. CUDA Toolkit: 12.8. Triton: 3.5.0
\        /    Bfloat16 = TRUE. FA [Xformers = 0.0.33.post1. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Unsloth 2025.12.8 patched 16 layers with 16 QKV layers, 16 O layers and 16 MLP layers.


⏳ Merging adapters...
Found HuggingFace hub cache directory: /home/david-barnes/.cache/huggingface/hub
Checking cache directory for required files...
Cache check failed: model.safetensors not found in local cache.
Not all required files found in cache. Will proceed with downloading.
Checking cache directory for required files...
Cache check failed: tokenizer.model not found in local cache.
Not all required files found in cache. Will proceed with downloading.


Unsloth: Preparing safetensor model files: 100%|██████████| 1/1 [00:33<00:00, 33.81s/it]


Note: tokenizer.model not found (this is OK for non-SentencePiece models)


Unsloth: Merging weights into 16bit: 100%|██████████| 1/1 [00:05<00:00,  5.09s/it]


Unsloth: Merge process complete. Saved to `/home/david-barnes/Documents/Projects/heirarch/llama-1b-reasoning-merged`
✅ Merged model saved!


## Test the Trained Model

In [11]:
# Cell 10: Test Inference
from unsloth import FastLanguageModel

# Put model in inference mode
FastLanguageModel.for_inference(model)

# Test question
test_question = "A loan is repaid with 20 equal annual payments. The interest portion of the 16th payment is 400 and the interest portion of the 11th payment is 600. Find the interest portion of the 1st payment."

messages = [
    {"role": "system", "content": SYSTEM_PROMPT.strip()},
    {"role": "user", "content": test_question}
]

inputs = tokenizer.apply_chat_template(
    messages,
    tokenize=True,
    add_generation_prompt=True,
    return_tensors="pt"
).to(model.device)

outputs = model.generate(
    input_ids=inputs,
    max_new_tokens=256,
    temperature=0.7,
)

response = tokenizer.decode(outputs[0], skip_special_tokens=True)
print("Question:", test_question)
print("\nResponse:")
print(response)

Question: A loan is repaid with 20 equal annual payments. The interest portion of the 16th payment is 400 and the interest portion of the 11th payment is 600. Find the interest portion of the 1st payment.

Response:
system

Cutting Knowledge Date: December 2023
Today Date: 26 Dec 2025

You are a mathematical reasoning assistant. Think through problems step by step.
Respond in the following format:
<reasoning>
...
</reasoning>
<answer>
...
</answer>user

A loan is repaid with 20 equal annual payments. The interest portion of the 16th payment is 400 and the interest portion of the 11th payment is 600. Find the interest portion of the 1st payment.assistant

To find the interest portion of the 1st payment, we first need to determine the interest portion of the 15th payment. Since the loan is repaid with 20 equal annual payments, we can use the formula for the total amount repaid:

Total Repaid = Principal + Interest

We know the interest portion of the 16th payment is 400 and the interest 

# Bench for unsloth_HICRA

In [14]:
import lm_eval
from lm_eval.models.huggingface import HFLM

# 1. Load your local model
# If you just saved your model to "llama-1b-reasoning-final", point to that folder.
print("⏳ Loading model for evaluation...")

# We wrap the model in the Harness's HFLM wrapper
# 'pretrained' can be a local path OR a Hub ID (e.g., "david-barnes/my-model")
llm = HFLM(
    pretrained="llama-1b-reasoning-merged",  # Use the merged model
    batch_size=1,
    trust_remote_code=True,
    dtype="bfloat16"
)

# 2. Define the tasks you want
# These key names correspond to the harness registry.
# Note: "minerva_math" is often split by subject (algebra, etc), 
# so we usually run the main "math" group or specific subtasks.
task_list = [
    "aime24",          # AIME 2024
    "minerva_math",    # Minerva Math (covers multiple subjects)
    # "math_500",        # The 'easy' 500 questions from MATH
    "leaderboard_gpqa_main",   # leaderboard_math_hard      
]

# 3. Run the Eval
print(f"🚀 Running evaluation on: {task_list}...")
results = lm_eval.simple_evaluate(
    model=llm,
    tasks=task_list,
    num_fewshot=0,        # Reasoning models often prefer 0-shot (Instruction)
    limit=None,           # Set to e.g., 50 to test quickly before full run!
    log_samples=True,    # Set True if you want to see exactly what it got wrong
)

# 4. Print a Pretty Table
from lm_eval.utils import make_table
print(make_table(results))

# 5. Save detailed results to JSON (Crucial for your blog!)
import json
with open("llama_1b_unsloth_HICRA_v1_benchmark_results.json", "w") as f:
    json.dump(results, f, indent=2)

[2025-12-26 10:43:01] INFO huggingface.py:158: Using device 'cuda'


⏳ Loading model for evaluation...


The tokenizer you are loading from 'llama-1b-reasoning-merged' with an incorrect regex pattern: https://huggingface.co/mistralai/Mistral-Small-3.1-24B-Instruct-2503/discussions/84#69121093e8b480e709447d5e. This will lead to incorrect tokenization. You should set the `fix_mistral_regex=True` flag when loading this tokenizer to fix this issue.
[2025-12-26 10:43:02] INFO huggingface.py:420: Model parallel was set to False, max memory was not set, and device map was set to {'': 'cuda'}
[2025-12-26 10:43:02] INFO evaluator.py:202: Setting random seed to 0 | Setting numpy seed to 1234 | Setting torch manual seed to 1234 | Setting fewshot manual seed to 1234
[2025-12-26 10:43:02] INFO evaluator.py:258: Using pre-initialized model


🚀 Running evaluation on: ['aime24', 'minerva_math', 'leaderboard_gpqa_main']...


Map: 100%|██████████| 448/448 [00:00<00:00, 2259.56 examples/s]
[2025-12-26 10:43:15] INFO __init__.py:695: Selected tasks:
[2025-12-26 10:43:15] INFO __init__.py:686: Task: leaderboard_gpqa_main (leaderboard/gpqa/gpqa_main_zeroshot.yaml)
[2025-12-26 10:43:15] INFO __init__.py:698: Group: minerva_math
[2025-12-26 10:43:15] INFO __init__.py:712: ConfigurableGroup(group=minerva_math,group_alias=None): {'minerva_math_algebra': ConfigurableTask(task_name=minerva_math_algebra,output_type=generate_until,num_fewshot=4,num_samples=1187), 'minerva_math_counting_and_prob': ConfigurableTask(task_name=minerva_math_counting_and_prob,output_type=generate_until,num_fewshot=4,num_samples=474), 'minerva_math_geometry': ConfigurableTask(task_name=minerva_math_geometry,output_type=generate_until,num_fewshot=4,num_samples=479), 'minerva_math_intermediate_algebra': ConfigurableTask(task_name=minerva_math_intermediate_algebra,output_type=generate_until,num_fewshot=4,num_samples=903), 'minerva_math_num_theor

KeyboardInterrupt: 

# Soft VRAM clear

In [13]:
import torch
import gc

# 1. Delete the Python variables holding the model
# (Wrap in try/except so it doesn't crash if they are already gone)
try:
    del model
    del tokenizer
    del trainer
except NameError:
    print("Variables already deleted or not defined.")

# 2. Python Garbage Collection (Clears CPU RAM)
gc.collect()

# 3. PyTorch Cache Clearing (The most important step for VRAM)
torch.cuda.empty_cache()

# Verify: Print current memory usage
print(f"GPU Memory Allocated: {torch.cuda.memory_allocated() / 1024**3:.2f} GB")
print(f"GPU Memory Reserved:  {torch.cuda.memory_reserved() / 1024**3:.2f} GB")

Variables already deleted or not defined.
GPU Memory Allocated: 2.30 GB
GPU Memory Reserved:  3.69 GB
